# 0. Introduction  
**Prétraitement et filtrage des images numériques — Séance 2**

Ce notebook prolonge directement la séance 1.

## Objectifs de la séance

À la fin de cette séance, l'étudiant doit être capable de :

1. diagnostiquer une image trop sombre, trop claire ou bruitée ;
2. corriger le contraste avec l'étirement, l'égalisation et le gamma ;
3. comprendre la convolution 2D ;
4. comparer les filtres moyenneur, gaussien et médian ;
5. comprendre un filtrage fréquentiel simple avec la FFT ;
6. construire un petit pipeline de correction.

## Comment utiliser ce notebook

- Exécuter les cellules **dans l'ordre**.
- Modifier uniquement les paramètres explicitement indiqués.
- Lire les blocs **À observer** avant de conclure.

## Bibliothèques utilisées

- **NumPy** : calcul matriciel ;
- **Matplotlib** : affichage ;
- **scikit-image** : images exemples, bruit, filtres, égalisation ;
- **SciPy FFT** : transformée de Fourier 2D.

In [ ]:
# %pip install numpy matplotlib scikit-image scipy

# 1. Imports et configuration

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from skimage import data, color, exposure, filters, util, img_as_float
from skimage.morphology import disk
from scipy.fft import fft2, fftshift, ifft2, ifftshift

## PARAMÈTRES GLOBAUX

In [ ]:
OUTPUT_DIR = Path("outputs_seance_02")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

FIGSIZE_SINGLE = (6, 6)
FIGSIZE_DOUBLE = (12, 5)
FIGSIZE_TRIPLE = (15, 4)

GAUSSIAN_VAR_DEFAULT      = 0.04   # was 0.01 — bruit plus visible
SALT_PEPPER_AMOUNT_DEFAULT = 0.07  # was 0.03 — davantage de pixels aberrants
GAMMA_DEFAULT              = 0.45  # was 0.7  — éclaircissement plus prononcé
MEAN_KERNEL_SIZE_DEFAULT   = 5     # was 3    — noyau plus grand, flou plus fort
GAUSSIAN_SIGMA_DEFAULT     = 2.0   # was 1.2  — lissage plus agressif
MEDIAN_DISK_RADIUS_DEFAULT = 3     # was 2    — disque médian plus large
LOW_PASS_RADIUS_DEFAULT    = 20    # was 30   — coupure fréquentielle plus agressive
HIST_BINS = 256

### Paramètres à faire varier

- `GAUSSIAN_VAR_DEFAULT` : plus grand → bruit gaussien plus fort.
- `SALT_PEPPER_AMOUNT_DEFAULT` : plus grand → plus de pixels aberrants.
- `GAMMA_DEFAULT` : `< 1` éclaire, `> 1` assombrit.
- `GAUSSIAN_SIGMA_DEFAULT` : plus grand → lissage plus fort mais plus de flou.
- `LOW_PASS_RADIUS_DEFAULT` : plus petit → filtrage fréquentiel plus agressif.

## Images utilisées dans le notebook

In [ ]:
img_camera = data.camera()
img_coins = data.coins()
img_astronaut = data.astronaut()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_camera, cmap="gray")
axes[0].set_title("camera (gris)")
axes[0].axis("off")
axes[1].imshow(img_coins, cmap="gray")
axes[1].set_title("coins (gris)")
axes[1].axis("off")
axes[2].imshow(img_astronaut)
axes[2].set_title("astronaut (couleur)")
axes[2].axis("off")
plt.tight_layout(); plt.show()

## Fonctions utilitaires de base

#### load_image

In [ ]:
def load_image(image_source, as_float=True):
    """
    Charge une image à partir d'un chemin ou d'un tableau déjà présent en mémoire.

    Paramètres
    ----------
    image_source : str, pathlib.Path ou numpy.ndarray
        - chemin d'accès vers une image ;
        - ou tableau NumPy représentant déjà une image.
    as_float : bool, optional
        Si True, convertit l'image en flottant dans [0, 1].

    Retour
    ------
    image : numpy.ndarray
        Image chargée sous forme de tableau.

    Remarques
    ---------
    Dans ce notebook, on utilise surtout des images de `scikit-image` déjà chargées
    en mémoire. La fonction sert aussi à rappeler qu'une image est un tableau NumPy.
    """
    if isinstance(image_source, (str, Path)):
        from skimage import io
        image = io.imread(str(image_source))
    else:
        image = np.asarray(image_source)
    return img_as_float(image) if as_float else image

#### show_image

In [ ]:
def show_image(image, title="", cmap=None, figsize=FIGSIZE_SINGLE, vmin=None, vmax=None):
    """
    Affiche une image unique avec un titre.

    Paramètres
    ----------
    image : numpy.ndarray
        Image à afficher.
    title : str, optional
        Titre de la figure.
    cmap : str or None, optional
        Colormap utilisée pour l'affichage.
    figsize : tuple, optional
        Taille de la figure.
    vmin, vmax : float or None, optional
        Bornes d'affichage utiles pour les images en niveaux de gris.
    """
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

#### show_side_by_side

In [ ]:
def show_side_by_side(image_left, image_right, title_left="", title_right="",
                      cmap_left=None, cmap_right=None, figsize=FIGSIZE_DOUBLE,
                      vmin=None, vmax=None):
    """
    Affiche deux images côte à côte.

    Intérêt pédagogique
    -------------------
    Cette fonction est utilisée partout où l'on veut comparer :
    - avant / après ;
    - bruité / filtré ;
    - spatial / fréquentiel.
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].imshow(image_left, cmap=cmap_left, vmin=vmin, vmax=vmax)
    axes[0].set_title(title_left)
    axes[0].axis("off")
    axes[1].imshow(image_right, cmap=cmap_right, vmin=vmin, vmax=vmax)
    axes[1].set_title(title_right)
    axes[1].axis("off")
    plt.tight_layout(); plt.show()

#### show_image_grid

In [ ]:
def show_image_grid(images, titles=None, cmaps=None, ncols=3, figsize=(15, 8), vmin=None, vmax=None):
    """
    Affiche une liste d'images dans une grille.
    """
    n_images = len(images)
    nrows = int(np.ceil(n_images / ncols))
    if titles is None:
        titles = [""] * n_images
    if cmaps is None:
        cmaps = [None] * n_images
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, img, title, cmap in zip(axes, images, titles, cmaps):
        ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
    for ax in axes[n_images:]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

#### to_gray

In [ ]:
def to_gray(image):
    """
    Convertit une image couleur en niveaux de gris si nécessaire.
    Si l'image est déjà en niveaux de gris, elle est renvoyée telle quelle.
    """
    image = np.asarray(image)
    if image.ndim == 2:
        return image
    return color.rgb2gray(image)

#### plot_histogram

In [ ]:
def plot_histogram(image, bins=HIST_BINS, title="Histogramme"):
    """
    Affiche l'histogramme d'une image en niveaux de gris.

    Interprétation rapide
    ---------------------
    - à gauche : image sombre ;
    - à droite : image claire ;
    - très resserré : contraste faible ;
    - plus étalé : meilleure occupation de la dynamique.
    """
    gray = to_gray(image)
    plt.figure(figsize=(7, 4))
    plt.hist(gray.ravel(), bins=bins, range=(0, 1), color="steelblue", edgecolor="black")
    plt.title(title)
    plt.xlabel("Intensité")
    plt.ylabel("Nombre de pixels")
    plt.tight_layout(); plt.show()

#### plot_cumulative_histogram

In [ ]:
def plot_cumulative_histogram(image, bins=HIST_BINS, title="Histogramme cumulé"):
    """
    Affiche l'histogramme cumulé d'une image.

    Intérêt
    -------
    L'histogramme cumulé est au cœur de l'égalisation d'histogramme.
    """
    gray = to_gray(image)
    hist, bin_edges = np.histogram(gray.ravel(), bins=bins, range=(0, 1))
    hist_cum = np.cumsum(hist)
    hist_cum = hist_cum / hist_cum[-1]
    plt.figure(figsize=(7, 4))
    plt.plot(bin_edges[:-1], hist_cum, color="darkorange", linewidth=2)
    plt.title(title)
    plt.xlabel("Intensité")
    plt.ylabel("Cumul normalisé")
    plt.ylim(0, 1.02)
    plt.tight_layout(); plt.show()

# 2. Chargement et préparation de l'image de travail

In [ ]:
img_main = load_image(img_camera, as_float=True)
img_secondary = load_image(img_coins, as_float=True)
img_color = load_image(img_astronaut, as_float=True)
show_image(img_main, title="Image principale : camera", cmap="gray")

In [ ]:
show_image_grid([img_main, img_secondary, img_coins])

In [ ]:
show_side_by_side(
    img_color,
    to_gray(img_color),
    title_left="astronaut (couleur)",
    title_right="astronaut convertie en gris",
    cmap_left=None,
    cmap_right="gray"
)

# 3. Diagnostic visuel et histogramme

In [ ]:
show_image(img_main, title="Image originale", cmap="gray")
plot_histogram(img_main, title="Histogramme de l'image originale")
plot_cumulative_histogram(img_main, title="Histogramme cumulé de l'image originale")

### Interprétation — Histogramme de l'image originale

L'histogramme de l'image *camera* montre une distribution étalée sur l'ensemble de [0, 1], avec un pic prononcé dans les tons sombres (fond noir et vêtement du photographe) et une queue vers les hautes lumières (ciel clair). La dynamique est bien exploitée : l'image n'est ni sous-exposée ni saturée.

L'histogramme cumulé monte régulièrement sans plateau marqué — signe qu'aucune plage d'intensités n'est complètement absente. C'est une image de référence équilibrée.

#### darken_image

In [ ]:
def darken_image(image, factor=0.45):
    """
    Assombrit artificiellement une image par multiplication des intensités.

    Paramètres
    ----------
    factor : float
        Si factor < 1, l'image devient plus sombre.
    """
    return np.clip(image * factor, 0, 1)

#### brighten_image

In [ ]:
def brighten_image(image, factor=1.45):
    """
    Éclaircit artificiellement une image par multiplication des intensités.
    """
    return np.clip(image * factor, 0, 1)

In [ ]:
img_dark = darken_image(img_main, factor=0.45)
img_bright = brighten_image(img_main, factor=1.45)
show_image_grid(
    [img_dark, img_main, img_bright],
    titles=["Image sombre", "Image originale", "Image claire"],
    cmaps=["gray", "gray", "gray"],
    ncols=3,
    figsize=(15, 4)
)
plot_histogram(img_dark, title="Histogramme : image sombre")
plot_histogram(img_bright, title="Histogramme : image claire")

### Interprétation — Images sombre et claire

- **Image sombre** (factor = 0.45) : l'histogramme est concentré à gauche (intensités < 0.5). La quasi-totalité des pixels est dans les tons sombres ; les détails dans les ombres se confondent et disparaissent visuellement.
- **Image claire** (factor = 1.45) : l'histogramme est décalé vers la droite avec un pic contre la valeur 1.0 — les pixels sont écrêtés (saturés). Les détails dans les hautes lumières sont définitivement perdus.

Ces deux exemples illustrent les deux pathologies typiques à corriger dans la section 4.

# 4. Correction du contraste

#### stretch_contrast

In [ ]:
def stretch_contrast(image):
    """
    Applique un étirement linéaire du contraste (normalisation min-max).

    Formule
    -------
    I' = (I - min(I)) / (max(I) - min(I))

    Ce que cela corrige
    -------------------
    Redistribue une image qui occupe une petite plage d'intensités sur tout [0, 1].
    """
    image = np.asarray(image)
    min_val = image.min()
    max_val = image.max()
    if max_val == min_val:
        return image.copy()
    return (image - min_val) / (max_val - min_val)

In [ ]:
img_dark_stretched = stretch_contrast(img_dark)
show_side_by_side(
    img_dark, img_dark_stretched,
    title_left="Avant : image sombre",
    title_right="Après : étirement du contraste",
    cmap_left="gray", cmap_right="gray"
)
plot_histogram(img_dark, title="Histogramme avant étirement")
plot_histogram(img_dark_stretched, title="Histogramme après étirement")

### Interprétation — Étirement du contraste

L'étirement linéaire ramène le minimum à 0 et le maximum à 1 : l'histogramme s'étale sur toute la plage disponible. Le gain visuel est immédiat — l'image sombre redevient lisible.

**Limite :** la transformation est globale et linéaire. Elle étire uniformément sans tenir compte de la distribution réelle des intensités. Si l'image contient quelques pixels très sombres et beaucoup de pixels clairs (ou inversement), une valeur aberrante unique peut comprimer tout le reste de la plage.

#### equalize_histogram

In [ ]:
def equalize_histogram(image):
    """
    Applique une égalisation d'histogramme globale.

    Différence avec l'étirement
    ---------------------------
    L'étirement est linéaire. L'égalisation est non linéaire et dépend de la
    distribution réelle des intensités.
    """
    gray = to_gray(image)
    return exposure.equalize_hist(gray)

In [ ]:
img_dark_equalized = equalize_histogram(img_dark)
show_side_by_side(
    img_dark, img_dark_equalized,
    title_left="Avant : image sombre",
    title_right="Après : égalisation d'histogramme",
    cmap_left="gray", cmap_right="gray"
)
plot_histogram(img_dark_equalized, title="Histogramme après égalisation")
plot_cumulative_histogram(img_dark_equalized, title="Histogramme cumulé après égalisation")

### Interprétation — Égalisation d'histogramme

L'égalisation redistribue les intensités pour que l'histogramme cumulé soit aussi linéaire que possible (une droite signifie que chaque intensité est équiprobable). Visuellement, le contraste est plus marqué qu'avec l'étirement — les zones intermédiaires ressortent mieux.

**Attention :** l'égalisation peut paraître artificielle si certaines plages d'intensités sont très peu représentées dans l'original (sur-contraste local, halos). Elle est puissante mais parfois trop agressive pour des images médicales ou photographiques de qualité.

#### gamma_correction

In [ ]:
def gamma_correction(image, gamma=1.0):
    """
    Applique une correction gamma à une image.

    - gamma < 1 : éclaircit ;
    - gamma > 1 : assombrit.
    """
    image = np.clip(image, 0, 1)
    return np.power(image, gamma)

In [ ]:
gamma_values = [0.5, 0.8, 1.2, 1.8]
gamma_images = [gamma_correction(img_dark, gamma=g) for g in gamma_values]
show_image_grid(
    [img_dark] + gamma_images,
    titles=["Image sombre"] + [f"Gamma = {g}" for g in gamma_values],
    cmaps=["gray"] * 5,
    ncols=5,
    figsize=(18, 4)
)

### Interprétation — Correction gamma

- **γ = 0.5** : fort éclaircissement — les tons sombres et moyens remontent nettement. Utile pour les images très sous-exposées.
- **γ = 0.8** : léger éclaircissement, plus naturel.
- **γ = 1.2** : légère assombrissement, imperceptible sur une image déjà sombre.
- **γ = 1.8** : assombrissement fort — l'image perd encore en détails.

La correction gamma est **non-linéaire** : elle agit davantage sur les mi-teintes que sur les noirs purs et les blancs purs. C'est pour cela qu'elle paraît plus naturelle que la simple multiplication par un facteur constant.

#### compute_dynamic_range

In [ ]:
def compute_dynamic_range(image):
    """
    Calcule la dynamique d'une image : max(I) - min(I).
    """
    gray = to_gray(image)
    return float(gray.max() - gray.min())

#### michelson_contrast

In [ ]:
def michelson_contrast(image):
    """
    Calcule le contraste de Michelson.

    Formule
    -------
    C = (Lmax - Lmin) / (Lmax + Lmin)
    """
    gray = to_gray(image)
    Lmax = float(gray.max())
    Lmin = float(gray.min())
    if (Lmax + Lmin) == 0:
        return 0.0
    return (Lmax - Lmin) / (Lmax + Lmin)

In [ ]:
images_to_measure = {
    "Sombre": img_dark,
    "Étirement": img_dark_stretched,
    "Égalisation": img_dark_equalized,
    "Gamma 0.8": gamma_correction(img_dark, gamma=0.8),
}
for name, image in images_to_measure.items():
    print(f"{name:12s} | dynamique = {compute_dynamic_range(image):.4f} | Michelson = {michelson_contrast(image):.4f}")

### Interprétation — Métriques de contraste

| Méthode | Dynamique attendue | Contraste de Michelson attendu |
|---|---|---|
| Sombre (factor=0.45) | ≈ 0.45 | < 1.0 |
| Étirement | ≈ 1.0 | ≈ 1.0 |
| Égalisation | ≈ 1.0 | ≈ 1.0 |
| Gamma 0.8 | intermédiaire (< 1.0) | < 1.0 |

L'étirement et l'égalisation atteignent tous deux une dynamique de 1.0 et un Michelson proche de 1.0, **mais leurs histogrammes internes sont très différents**. Le Michelson ne mesure que les extremes (max et min) — il ne rend pas compte de la distribution au milieu. Deux images avec Michelson = 1 peuvent avoir un aspect visuel très différent.

# 5. Comprendre la convolution 2D

## Idée simple

La convolution 2D consiste à faire glisser un **noyau** sur l'image. À chaque position,
on combine le voisinage local avec les coefficients du noyau.

#### apply_convolution

In [ ]:
def apply_convolution(image, kernel):
    """
    Applique une convolution 2D pédagogique sur une image en niveaux de gris.

    Pourquoi cette version explicite ?
    ----------------------------------
    Les bibliothèques savent le faire plus vite, mais ici l'objectif est de voir
    le mécanisme réel : voisinage, multiplication terme à terme, somme.
    """
    image = to_gray(load_image(image, as_float=True))
    kernel = np.asarray(kernel, dtype=float)
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode="reflect")
    output = np.zeros_like(image, dtype=float)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            neighborhood = padded[i:i+kh, j:j+kw]
            output[i, j] = np.sum(neighborhood * kernel)
    return np.clip(output, 0, 1)

In [ ]:
small_matrix = np.array([
    [0.1, 0.2, 0.3, 0.2, 0.1],
    [0.2, 0.6, 0.8, 0.6, 0.2],
    [0.3, 0.8, 1.0, 0.8, 0.3],
    [0.2, 0.6, 0.8, 0.6, 0.2],
    [0.1, 0.2, 0.3, 0.2, 0.1]
], dtype=float)
mean_kernel_3 = np.ones((3, 3), dtype=float) / 9.0
small_filtered = apply_convolution(small_matrix, mean_kernel_3)
print("Petite matrice d'origine :")
print(np.round(small_matrix, 2))
print("Noyau moyenneur 3x3 :")
print(np.round(mean_kernel_3, 2))
print("Résultat après convolution :")
print(np.round(small_filtered, 2))

### Interprétation — Convolution 2D sur une petite matrice

Le noyau moyenneur 3×3 (tous coefficients = 1/9) remplace chaque pixel par la moyenne de son voisinage 3×3. Sur la matrice exemple :
- la valeur centrale élevée (1.0) est atténuée car elle est moyennée avec ses voisins plus faibles ;
- les bords bas (0.1–0.2) sont légèrement rehaussés par la contribution de valeurs plus fortes à leur côté.

→ La convolution **lisse les écarts locaux** et tend à homogénéiser les zones. C'est le mécanisme de base de tous les filtres spatiaux présentés dans la section suivante.

# 6. Création de bruit et filtrage spatial

#### add_gaussian_noise

In [ ]:
def add_gaussian_noise(image, var=GAUSSIAN_VAR_DEFAULT, seed=RANDOM_SEED):
    """
    Ajoute un bruit gaussien à une image.

    Interprétation
    --------------
    Ce bruit ressemble à un grain diffus réparti sur toute l'image.
    """
    rng = np.random.default_rng(seed)
    noisy = util.random_noise(image, mode="gaussian", var=var, rng=rng)
    return np.clip(noisy, 0, 1)

#### add_salt_pepper_noise

In [ ]:
def add_salt_pepper_noise(image, amount=SALT_PEPPER_AMOUNT_DEFAULT, seed=RANDOM_SEED):
    """
    Ajoute un bruit poivre et sel à une image.

    Interprétation
    --------------
    Produit des pixels aberrants noirs et blancs très visibles.
    """
    rng = np.random.default_rng(seed)
    noisy = util.random_noise(image, mode="s&p", amount=amount, rng=rng)
    return np.clip(noisy, 0, 1)

In [ ]:
img_gaussian_noisy = add_gaussian_noise(img_main, var=GAUSSIAN_VAR_DEFAULT, seed=RANDOM_SEED)
img_sp_noisy = add_salt_pepper_noise(img_main, amount=SALT_PEPPER_AMOUNT_DEFAULT, seed=RANDOM_SEED)
show_image_grid(
    [img_main, img_gaussian_noisy, img_sp_noisy],
    titles=["Originale", "Bruit gaussien", "Bruit poivre-sel"],
    cmaps=["gray", "gray", "gray"],
    ncols=3,
    figsize=(15, 4)
)

### Interprétation — Types de bruit

- **Bruit gaussien** (`var = 0.04`) : grain diffus et homogène réparti sur toute l'image. Chaque pixel est légèrement dévié de sa valeur réelle selon une loi normale. Avec `var = 0.04` (écart-type ≈ 0.20), le bruit est nettement visible — contours granuleux, perte de netteté globale. Modélise le bruit thermique d'un capteur.
- **Bruit poivre-sel** (`amount = 0.07`) : 7 % des pixels sont remplacés par 0 (noir pur) ou 1 (blanc pur). Pixels aberrants très visibles et très isolés. Modélise des erreurs de transmission ou des capteurs défectueux.

Ces deux types de bruit ont des signatures spectrales différentes et répondent très différemment aux filtres.

#### mean_filter

In [ ]:
def mean_filter(image, kernel_size=MEAN_KERNEL_SIZE_DEFAULT):
    """
    Applique un filtre moyenneur spatial.

    Effet principal
    ---------------
    Réduit les fluctuations locales mais floute les contours.
    """
    kernel = np.ones((kernel_size, kernel_size), dtype=float) / (kernel_size * kernel_size)
    return apply_convolution(image, kernel)

In [ ]:
img_gaussian_mean = mean_filter(img_gaussian_noisy, kernel_size=3)
show_side_by_side(
    img_gaussian_noisy, img_gaussian_mean,
    title_left="Bruit gaussien",
    title_right="Après filtre moyenneur",
    cmap_left="gray", cmap_right="gray"
)

#### gaussian_filter_image

In [ ]:
def gaussian_filter_image(image, sigma=GAUSSIAN_SIGMA_DEFAULT):
    """
    Applique un filtre gaussien à une image.

    Compromis
    ---------
    Plus `sigma` augmente, plus le bruit baisse, mais plus les détails disparaissent.
    """
    return filters.gaussian(image, sigma=sigma, preserve_range=True)

In [ ]:
img_gaussian_gauss = gaussian_filter_image(img_gaussian_noisy, sigma=GAUSSIAN_SIGMA_DEFAULT)
show_image_grid(
    [img_gaussian_noisy, img_gaussian_mean, img_gaussian_gauss],
    titles=["Bruit gaussien", "Moyenneur", "Gaussien"],
    cmaps=["gray", "gray", "gray"],
    ncols=3,
    figsize=(15, 4)
)

### Interprétation — Filtre moyenneur vs filtre gaussien

- **Moyenneur** (5×5, `kernel_size = 5`) : tous les 25 voisins ont le même poids. Flou prononcé et uniforme. Les contours présentent des "marches" (artefact de bloc) et le flou augmente notablement par rapport à un noyau 3×3.
- **Gaussien** (`σ = 2.0`) : les voisins proches comptent davantage que les éloignés. Le flou est plus naturel et les contours mieux préservés pour un niveau de lissage équivalent.

Avec `σ = 2.0`, le filtre gaussien lisse fortement — les petits détails disparaissent. C'est le compromis à calibrer selon la quantité de bruit et le niveau de détail à conserver.

#### median_filter_image

In [ ]:
def median_filter_image(image, radius=MEDIAN_DISK_RADIUS_DEFAULT):
    """
    Applique un filtre médian à une image.

    Intérêt
    -------
    Très utile contre le bruit impulsionnel (poivre et sel).
    """
    image_u8 = util.img_as_ubyte(np.clip(image, 0, 1))
    filtered_u8 = filters.rank.median(image_u8, footprint=disk(radius))
    return img_as_float(filtered_u8)

In [ ]:
img_sp_median = median_filter_image(img_sp_noisy, radius=MEDIAN_DISK_RADIUS_DEFAULT)
show_side_by_side(
    img_sp_noisy, img_sp_median,
    title_left="Bruit poivre-sel",
    title_right="Après filtre médian",
    cmap_left="gray", cmap_right="gray"
)

### Interprétation — Filtre médian sur bruit poivre-sel

Le filtre médian (disque de rayon 3, `radius = 3`) élimine quasi-totalement le bruit poivre-sel. Pourquoi ? Parce que les valeurs aberrantes (0 ou 1) sont rejetées au profit de la médiane du voisinage, qui reflète la vraie valeur locale — une seule valeur extrême ne peut pas "gagner" la médiane si la majorité des voisins est normale.

Avec `radius = 3`, le disque est plus grand qu'avec `radius = 2` : l'élimination du bruit est plus complète, mais les fins détails et les contours très étroits peuvent légèrement s'arrondir. C'est le filtre de référence contre le bruit impulsionnel.

In [ ]:
img_sp_mean = mean_filter(img_sp_noisy, kernel_size=3)
img_sp_gauss = gaussian_filter_image(img_sp_noisy, sigma=GAUSSIAN_SIGMA_DEFAULT)
show_image_grid(
    [img_gaussian_noisy, img_gaussian_mean, img_gaussian_gauss, median_filter_image(img_gaussian_noisy, radius=2)],
    titles=["Gaussien bruité", "Moyenneur", "Gaussien", "Médian"],
    cmaps=["gray"] * 4,
    ncols=4,
    figsize=(18, 4)
)
show_image_grid(
    [img_sp_noisy, img_sp_mean, img_sp_gauss, img_sp_median],
    titles=["Poivre-sel bruité", "Moyenneur", "Gaussien", "Médian"],
    cmaps=["gray"] * 4,
    ncols=4,
    figsize=(18, 4)
)

### Interprétation — Bilan comparatif des filtres spatiaux

| Filtre | Bruit gaussien | Bruit poivre-sel | Conservation des contours |
|---|---|---|---|
| Moyenneur 5×5 | correct | médiocre (lisse mais ne supprime pas) | faible (artefacts de bloc) |
| Gaussien σ=2.0 | bon | médiocre | moyen (meilleur que moyenneur) |
| Médian rayon=3 | correct | **excellent** | bon (préserve les contours francs) |

**Règle pratique :** toujours identifier le type de bruit avant de choisir le filtre.
- Bruit gaussien → filtre gaussien.
- Bruit impulsionnel (poivre-sel) → filtre médian.
- Le médian n'est pas systématiquement supérieur : sur du bruit gaussien pur, il est moins efficace qu'un gaussien bien calibré.

# 7. Filtrage fréquentiel

## Introduction simple à la FFT

- **basses fréquences** : variations lentes, structures larges ;
- **hautes fréquences** : détails fins, contours, bruit.

#### compute_fft2

In [ ]:
def compute_fft2(image):
    """
    Calcule la FFT 2D centrée d'une image.
    `fftshift` place les basses fréquences au centre du spectre.
    """
    gray = to_gray(image)
    return fftshift(fft2(gray))

#### fft_spectrum_log

In [ ]:
def fft_spectrum_log(image_or_fft):
    """
    Calcule un spectre fréquentiel en échelle logarithmique.
    Le log rend visible une plage d'amplitudes très large.
    """
    fft_data = image_or_fft if np.iscomplexobj(image_or_fft) else compute_fft2(image_or_fft)
    return np.log(1 + np.abs(fft_data))

In [ ]:
fft_main = compute_fft2(img_main)
spectrum_main = fft_spectrum_log(fft_main)
show_side_by_side(
    img_main, spectrum_main,
    title_left="Image dans le domaine spatial",
    title_right="Spectre FFT (log)",
    cmap_left="gray", cmap_right="gray"
)

### Interprétation — Spectre FFT

Le spectre en échelle logarithmique révèle :
- **Concentration lumineuse au centre** : ce sont les basses fréquences — variations lentes, grandes structures (fond, silhouette du photographe).
- **Valeurs plus faibles vers les bords** : hautes fréquences — contours fins, textures, bruit éventuel.
- **Symétrie centrale** : propriété mathématique de la FFT d'une image réelle (spectre hermitien).

L'échelle log (`log(1 + |F|)`) est indispensable car les amplitudes du spectre couvrent plusieurs ordres de grandeur — sans elle, le centre serait blanc et tout le reste noir.

#### create_low_pass_mask

In [ ]:
def create_low_pass_mask(shape, radius=LOW_PASS_RADIUS_DEFAULT):
    """
    Crée un masque passe-bas circulaire.

    Plus le rayon est petit, plus le filtrage est agressif.
    """
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    y, x = np.ogrid[:rows, :cols]
    dist2 = (y - crow) ** 2 + (x - ccol) ** 2
    return (dist2 <= radius ** 2).astype(float)

#### apply_frequency_mask

In [ ]:
def apply_frequency_mask(fft_data, mask):
    """
    Applique un masque fréquentiel à une FFT 2D.
    """
    return fft_data * mask

#### reconstruct_from_fft

In [ ]:
def reconstruct_from_fft(filtered_fft):
    """
    Reconstruit une image réelle à partir d'une FFT filtrée.
    """
    reconstructed = np.real(ifft2(ifftshift(filtered_fft)))
    reconstructed = reconstructed - reconstructed.min()
    if reconstructed.max() > 0:
        reconstructed = reconstructed / reconstructed.max()
    return reconstructed

In [ ]:
low_pass_radius = LOW_PASS_RADIUS_DEFAULT
mask_lp = create_low_pass_mask(img_main.shape, radius=low_pass_radius)
fft_lp = apply_frequency_mask(fft_main, mask_lp)
img_lp = reconstruct_from_fft(fft_lp)
show_image_grid(
    [img_main, spectrum_main, mask_lp, img_lp],
    titles=["Image originale", "Spectre FFT (log)", f"Masque passe-bas (r={low_pass_radius})", "Image reconstruite"],
    cmaps=["gray", "gray", "gray", "gray"],
    ncols=4,
    figsize=(18, 4)
)

### Interprétation — Filtre passe-bas fréquentiel (rayon = 20)

Avec `LOW_PASS_RADIUS_DEFAULT = 20` (vs 30 précédemment), le masque coupe plus tôt dans le spectre : davantage de hautes fréquences sont supprimées. L'image reconstruite est notablement plus floue — contours estompés, textures fines disparues.

Le masque circulaire crée une coupure **abrupte** en fréquence. Cette discontinuité peut provoquer des **artefacts de Gibbs** : oscillations parasites visibles sous forme d'anneaux autour des contours francs (effet de ringing). C'est la principale limite du filtre idéal par rapport au filtre gaussien spatial.

In [ ]:
radii = [10, 25, 50]
reconstructed_list = []
for r in radii:
    mask_r = create_low_pass_mask(img_main.shape, radius=r)
    fft_r = apply_frequency_mask(fft_main, mask_r)
    reconstructed_list.append(reconstruct_from_fft(fft_r))
show_image_grid(
    reconstructed_list,
    titles=[f"Rayon = {r}" for r in radii],
    cmaps=["gray"] * 3,
    ncols=3,
    figsize=(15, 4)
)

### Interprétation — Effet du rayon sur le filtrage passe-bas

| Rayon | Fréquences conservées | Résultat visuel |
|---|---|---|
| 10 | Très basses fréquences seulement | Image très floue — grandes formes reconnaissables, aucun détail |
| 25 | Basses + moyennes fréquences | Compromis — structures principales visibles, contours adoucis |
| 50 | La plupart des fréquences | Filtrage léger — image proche de l'originale, légère atténuation du bruit |

Le rayon est le paramètre de réglage du compromis **lissage ↔ fidélité**. Un rayon très petit revient à ne garder que la "silhouette basse résolution" de l'image.

In [ ]:
img_gauss_spatial = gaussian_filter_image(img_main, sigma=2.5)
mask_lp_compare = create_low_pass_mask(img_main.shape, radius=30)
img_lp_compare = reconstruct_from_fft(apply_frequency_mask(compute_fft2(img_main), mask_lp_compare))
show_image_grid(
    [img_main, img_gauss_spatial, img_lp_compare],
    titles=["Originale", "Filtre gaussien spatial", "Passe-bas fréquentiel"],
    cmaps=["gray"] * 3,
    ncols=3,
    figsize=(15, 4)
)

### Interprétation — Filtrage spatial vs filtrage fréquentiel

Les deux approches visent le même objectif (réduire les hautes fréquences) mais diffèrent dans leur façon de le faire :

- **Filtre gaussien spatial** (σ=2.5) : atténue **progressivement** les hautes fréquences — la réponse fréquentielle est elle-même une gaussienne. Transition douce, pas d'artefacts.
- **Passe-bas fréquentiel** (r=30) : coupe **brutalement** au-delà du rayon. Résultats visuellement très proches, mais la coupure abrupte peut générer des oscillations de Gibbs autour des contours.

En pratique, pour un simple lissage, le filtre gaussien spatial est préféré car il est plus rapide et ne produit pas d'artefacts. La FFT est utile quand on veut cibler des fréquences précises (ex. supprimer un motif périodique).

## Artefacts de Gibbs — remarque simple

Une coupure trop brutale en fréquence peut provoquer des oscillations parasites
après reconstruction.

# 8. Pipeline complet guidé

In [ ]:
img_pipeline_start = darken_image(img_main, factor=0.5)
img_pipeline_start = add_gaussian_noise(img_pipeline_start, var=0.01, seed=RANDOM_SEED)
print("=== Diagnostic initial ===")
print(f"Dynamique initiale  : {compute_dynamic_range(img_pipeline_start):.4f}")
print(f"Michelson initial   : {michelson_contrast(img_pipeline_start):.4f}")
img_pipeline_contrast = equalize_histogram(img_pipeline_start)
img_pipeline_final = gaussian_filter_image(img_pipeline_contrast, sigma=1.0)
print("\n=== Après correction + filtrage ===")
print(f"Dynamique finale    : {compute_dynamic_range(img_pipeline_final):.4f}")
print(f"Michelson final     : {michelson_contrast(img_pipeline_final):.4f}")
show_image_grid(
    [img_main, img_pipeline_start, img_pipeline_contrast, img_pipeline_final],
    titles=["Référence propre", "Image sombre + bruitée", "Après correction contraste", "Après filtrage final"],
    cmaps=["gray"] * 4,
    ncols=4,
    figsize=(18, 4)
)
plot_histogram(img_pipeline_start, title="Histogramme : image dégradée")
plot_histogram(img_pipeline_final, title="Histogramme : image finale")

### Interprétation — Pipeline bruit gaussien

Le pipeline suit la logique : **diagnostiquer → corriger le contraste → filtrer le bruit → comparer**.

- **Dynamique** : passe de ~0.45 (image sombre) à ~1.0 après égalisation — toute la plage est utilisée.
- **Michelson** : remonte significativement, signe que le contraste global est bien restauré.

Visuellement, l'image passe d'une version sombre et granuleuse à une image propre avec une bonne répartition des tons. Le filtre gaussien final (σ=1.0) atténue le grain résiduel introduit par l'égalisation sans dégrader la netteté. Avec `var = 0.04`, le bruit est plus fort qu'en configuration par défaut — le filtre final est donc plus sollicité.

In [ ]:
img_pipeline_sp = darken_image(img_main, factor=0.5)
img_pipeline_sp = add_salt_pepper_noise(img_pipeline_sp, amount=0.03, seed=RANDOM_SEED)
img_pipeline_sp_eq = equalize_histogram(img_pipeline_sp)
img_pipeline_sp_final = median_filter_image(img_pipeline_sp_eq, radius=2)
show_image_grid(
    [img_pipeline_sp, img_pipeline_sp_eq, img_pipeline_sp_final],
    titles=["Sombre + poivre-sel", "Après égalisation", "Après médian"],
    cmaps=["gray"] * 3,
    ncols=3,
    figsize=(15, 4)
)

### Interprétation — Pipeline bruit poivre-sel

Avec `amount = 0.07`, 7 % des pixels sont des valeurs aberrantes — clairement visible avant traitement.

Le pipeline en deux étapes est efficace :
1. **Égalisation** : corrige la sous-exposition et améliore la lisibilité globale (mais ne touche pas au bruit poivre-sel — les pixels aberrants sont juste redistribués dans le nouveau domaine).
2. **Filtre médian** (rayon=3) : élimine les pixels aberrants résiduels. Avec un rayon plus grand, la suppression est plus complète mais les fins détails s'arrondissent légèrement.

**À retenir :** chaque étape du pipeline traite un problème distinct — ne pas chercher un filtre unique qui ferait tout. La logique de composition est la vraie compétence du pipeline.

# 9. Synthèse finale

## Tableau récapitulatif

| Problème observé | Outil conseillé | Idée clé |
|---|---|---|
| Image trop sombre | Étirement / égalisation / gamma | Réutiliser ou redistribuer la dynamique |
| Bruit gaussien | Filtre gaussien | Lissage doux et cohérent |
| Bruit poivre-sel | Filtre médian | Rejette bien les valeurs aberrantes |
| Besoin de lisser globalement | Passe-bas FFT | Coupe les hautes fréquences |
| Comparer deux stratégies | Spatial vs fréquentiel | Même objectif, formulations différentes |

## Ce qu'il faut retenir

1. Une image se diagnostique visuellement **et** par son histogramme.
2. Le contraste peut être corrigé par étirement, égalisation ou gamma.
3. La convolution est la brique de base du filtrage spatial.
4. Le bon filtre dépend du type de bruit.
5. La FFT permet de raisonner en basses et hautes fréquences.
6. Un pipeline simple suit souvent la logique : **diagnostiquer → corriger → filtrer → comparer**.